# Ablation (large scale): few-shot only

`Abl-FewShotOnly` on `Air-NRM-CA-LS` and `Air-NRM-NP-LS`. Generated by `build_ls_ablations.py` from the full-arm notebook
`LEAN_LLM_OPT_4.1_Air_NRM_LargeScale.ipynb`: same classifier, same batch loop, same scoring,
same Final Answer template in the exemplar. Raw CSV text is attached through Responses input_file; no custom data retrieval is used.

In [ ]:
# GENERATED FILE -- do not edit by hand.
#
# Built by build_ls_ablations.py from
#   Test_Dataset/Air_NRM/large_scale/LEAN_LLM_OPT_4.1_Air_NRM_LargeScale.ipynb
# Arm: Abl-FewShotOnly on Air-NRM-CA-LS / Air-NRM-NP-LS. What this arm removes: custom retrieval (raw files supplied through API attachments).
# Edit the generator, re-run it, and commit both.

# ── large-scale fork ─────────────────────────────────────────────────────────
# A copy of LEAN_LLM_OPT_4.1_Air_NRM.ipynb, edited for the large_scale dataset
# (90 ODs, 268 departures, 4 fare families, per-flight capacity). The small-scale
# notebook at the repository root is untouched; the two are deliberately NOT
# merged for now.
#
# Every path below is written relative to the repository root, so move there
# first. This notebook lives three levels down, hence the three parents.
import os, sys
from pathlib import Path
_ROOT = Path.cwd()
while not (_ROOT / "exp_config.yaml").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
if not (_ROOT / "exp_config.yaml").exists():
    raise RuntimeError("run this notebook from inside the lean-llm-opt-main tree")
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))
print("working directory:", os.getcwd())


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
import pandas as pd
from langchain_classic.chains import RetrievalQA
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_core.documents import Document
from typing import List, Dict
from openai import OpenAI
import openai
from datetime import datetime
import re
from datetime import time as time_class
# ---------------------------------------------------------------------------
# Unified experiment configuration + run logging  (see PATCH_GUIDE.md)
# ---------------------------------------------------------------------------
import os
import leanopt_exp as lx

# API keys come from a .env file at the repository root (see .env.example),
# from the shell environment, or -- if neither is set -- from a hidden prompt.
# Nothing is hard-coded here and nothing is written to the run log.
try:
    from dotenv import load_dotenv
    # override=True: the .env file wins over anything already in the
    # environment. Without it, a stale key left in os.environ by an earlier
    # run silently shadows the value you just edited into .env.
    load_dotenv(override=True)
except ImportError:
    pass

CFG = lx.load_config(
    "exp_config.yaml",
    model_profile='gpt-4.1',
    method='Abl-FewShotOnly',
    dataset='Air-NRM-CA-LS',
    # Repeats. exp_config sets n_repeats: 3 but nothing was reading it here, so
    # every run landed in the same __r0__ bucket and overwrote the last. Taking
    # it from the environment keeps the default at 0 -- no existing invocation
    # changes -- while
    #     for r in 0 1 2; do LEANOPT_REPEAT=$r python run_notebook.py ...; done
    # produces three separately logged runs that aggregate_runs.py can pool into
    # mean +/- std. Worth doing here: temperature is 0.0 and the seed is fixed,
    # yet instance 13 was wrong in one run and right in the next, so a single
    # run cannot tell a stable failure from a coin flip.
    repeat_index=int(os.environ.get("LEANOPT_REPEAT", "0")),
)
lx.ensure_api_keys(CFG)                          # only for providers this profile uses
user_api_key = os.environ.get("OPENAI_API_KEY", "")   # legacy references

LOG = lx.RunLogger(CFG)
EMBEDDINGS = lx.build_embeddings(CFG)

## Problem classification

Unchanged from the full arm -- the classifier is not the component under test.

In [ ]:
def Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv"):
    # Initialize the LLM
    llm1 = lx.build_llm(CFG, "classifier")

    # Load and process the data
    # classification only needs the problem description + its class label
    refdata = lx.load_refdata_docs(CFG)
    refdocuments = refdata

    embeddings = EMBEDDINGS
    vectors1 = FAISS.from_documents(refdocuments, embeddings)
    retriever1 = lx.build_retriever(CFG, vectors1, "refdata")
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm1,
        chain_type="stuff",
        retriever=retriever1,
        return_source_documents=True,
    )

    qa_tool1 = Tool(
        name="FileQA",
        func=qa_chain.invoke,
        description=(
            "Use this tool to answer questions about the problem type of the text. "
            "Provide the question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
        ),
    )

    few_shot_examples = """

    Question: What is the problem type in operation of the text? Please give the answer directly. Text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1,w_2,w_3.There is an independent demand stream for each of the products. The objective of the company is to decide which demands to be fufilled over a ﬁnite sales horizon [0,10] to maximize the total expected revenue from ﬁxed initial inventories. The on-hand inventories for the three items are c_1,c_2,c_3 respectively. During the sales horizon, replenishment is not allowed and there is no any in-transit inventories. Customers who want to purchase P1,P2,P3 arrive at each period accoring to a Poisson process with a_1,a_2,a_3 the arrival rates respectively. Decision variables y_1,y_2,y_3 correspond to the number of requests that the firm plans to fulfill for product 1,2,3. These variables are all positive integers.

    Thought: I need to determine the problem type of the text. I'll use the FileQA tool to retrieve the relevant information.

    Action: FileQA

    Action Input: "What is the problem type in operation of the text? text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1, w_2, w_3. ..."

    Observation: The problem type of the text is Network Revenue Management.

    Final Answer: Network Revenue Management.

    """

    prefix = f"""You are a helpful assistant that can answer questions about operation problems.

    Use the following examples as a guide. Always use the FileQA tool when you need to retrieve information from the file:


    {few_shot_examples}

    When you need to find information from the file, use the provided tools. And answer the question by given the answer directly. For example,

    """

    suffix = """

    Begin!

    Question: {input}
    {agent_scratchpad}"""

    agent_pc = initialize_agent(
        tools=[qa_tool1],
        llm=llm1,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        **lx.agent_kwargs(CFG, prefix.format(few_shot_examples=few_shot_examples), suffix),
    )

    openai.api_request_timeout = 60 
    return agent_pc

def Problemtype(query):
    agent_pc = Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv")
    category_original=agent_pc.invoke(f"What is the problem type in operation of the text? text:{query}")
    type_output = category_original['output']
    return type_output

## Few-shot only

Attach complete raw CSV files through the API and keep the worked example.

In [ ]:
# LARGE-SCALE FORK: reads Test_Dataset/Air_NRM/large_scale/ and additionally
# loads flight_capacity.csv, which the small-scale case does not have (there,
# every flight is 187 seats and the number is hard-coded).
#
# CACHE 1 of 2 -- the CSVs. LoadFiles() is called from New_Vectors_Flight,
# New_Vectors_Demand and retrieve_parameter, i.e. several times per flight per
# query. Re-reading five CSVs each time is pure waste; they never change during
# a run. Same pattern as _SCHEDULE_CACHE in build_air_nrm_inputs.py.
_LS_DIR = 'Test_Dataset/Air_NRM/large_scale'
_FILES_CACHE = None


def LoadFiles():
  """v1, v2, od_demand, flight, flight_capacity -- read once, then reused."""
  global _FILES_CACHE
  if _FILES_CACHE is None:
      _FILES_CACHE = (
          pd.read_csv(f'{_LS_DIR}/v1.csv'),
          pd.read_csv(f'{_LS_DIR}/v2.csv'),
          pd.read_csv(f'{_LS_DIR}/od_demand.csv'),
          pd.read_csv(f'{_LS_DIR}/flight.csv'),
          pd.read_csv(f'{_LS_DIR}/flight_capacity.csv'),
      )
  return _FILES_CACHE


def LoadProducts():
  """Fare families, in the order v1.csv lists them.

  The small-scale notebook hard-codes Eco_flexi and Eco_lite. Here there are
  four, and reading them off the column names means the list can never drift
  from the data file.
  """
  v1 = LoadFiles()[0]
  seen = []
  for column in v1.columns:
      if '*' in column:
          name = column.split('*')[0]
          if name not in seen:
              seen.append(name)
  return seen


In [ ]:
# Files are attached through Responses input_file, never as bare ID strings.
from airnrm_api_files import ApiFileInput
API_FILES = ApiFileInput(CFG, role="data_agent")


In [ ]:
# The exemplar. Precomputed, not retrieved -- that is the point of this
# arm. The parameter block below is byte-for-byte what csv_qa_tool_CA
# returns for the two exemplar flights in the full arm; it is checked
# against the CSVs by build_ls_ablations.py --check.
EXEMPLAR_DOC = "avg_price={'(865678,01:30,f)': 898.85, '(865678,01:30,s)': 646.2, '(865678,01:30,v)': 494.22, '(865678,01:30,l)': 374.15, '(915865,10:35,f)': 995.75, '(915865,10:35,s)': 556.38, '(915865,10:35,v)': 411.27, '(915865,10:35,l)': 312.32}\nvalue_list={'(865678,01:30,f)': 0.028394, '(865678,01:30,s)': 0.05569, '(865678,01:30,v)': 0.045195, '(865678,01:30,l)': 0.042998, '(915865,10:35,f)': 0.018204, '(915865,10:35,s)': 0.021255, '(915865,10:35,v)': 0.017254, '(915865,10:35,l)': 0.030832}\nratio_list={'(865678,01:30,f)': 0.678261, '(865678,01:30,s)': 0.664991, '(865678,01:30,v)': 0.630688, '(865678,01:30,l)': 0.561085, '(915865,10:35,f)': 0.960781, '(915865,10:35,s)': 0.955126, '(915865,10:35,v)': 0.948269, '(915865,10:35,l)': 0.936171}\nvalue_0_list={'865678': 0.383933, '915865': 0.638646}\nratio_0_list={'865678': 1.783683, '915865': 1.025854}\navg_pax={'865678': 2279.18, '915865': 2294.45}\ncapacity_consum={'f': 2.0, 's': 1.6, 'v': 1.3, 'l': 1.0}\nflight_capacity={'(865678,01:30)': 589.96, '(915865,10:35)': 360.64}\n"

FEWSHOT_CA = '\nQuestion: Based on all flight ticket choices in \'flight.csv\' and \'od_demand.csv\' with attraction values in v1 and shadow attraction value ratios in v2, develop the SBLP(sales-based linear programming) formulation for among flights (OD = (\'865\', \'678\') AND Departure Time=\'01:30\'), (OD = (\'915\', \'865\') AND Departure Time=\'10:35\'),  that maximize the total revenue of flight ticket sales. The SBLP should include decision variables, objective function, balance constraints, scale constraints, nonnegative constraints and selection constraints. Each flight\'s capacity is 28% of its weekly economy capacity \'Y Seats/Week\' in \'flight_capacity.csv\', matched on (Oneway_OD, Departure Time) — the seven points of sale this dataset covers account for 28% of the economy cabin. Please consider that each Eco_flexi ticket consumes 2 units of flight capacity, each Eco_standard ticket consumes 1.6 units, each Eco_value ticket consumes 1.3 units and each Eco_lite ticket consumes 1 unit of capacity\nFinal Answer:\n### Objective Function\n\n\\[\n\\max \\quad \\sum_(l,k,j) avg\\_price[(l,k,j)] \\cdot x[(l,k,j)]\n\\]\n\n### Constraints\n\n#### 1. Capacity Constraints\n\nFor each flight (OD pair \\( l \\), departure time \\( k \\)), summed over every\nfare family \\( j \\). Both the per-family consumption and the right-hand side\nare looked up, not constants:\n\n\\[\n\\sum_j capacity\\_consum[j] \\cdot x[(l,k,j)] \\leq flight\\_capacity[(l,k)]\n\\]\n\n#### 2. Balance Constraints\n\nFor each OD pair \\( l \\):\n\n\\[\nratio\\_0\\_list[l] \\cdot x_o[l] + \\sum_k \\sum_j ratio\\_list[(l,k,j)] \\cdot x[(l,k,j)] = avg\\_pax[l]\n\\]\n\n#### 3. Scale Constraints\n\nFor each ticket option \\((l,k,j)\\):\n\n\\[\nx[(l,k,j)]/value\\_list[(l,k,j)] - x_o[l]/value\\_0\\_list[l] \\leq 0\n\\]\n\n#### 4. Nonnegativity Constraints\n\n\\[\nx[(l,k,j)] \\geq 0, \\quad x_o[l] \\geq 0\n\\]\n- x = avg_price.keys()\n- x_o = avg_pax.keys()\n\n### Generated Code\n\n```python\nimport gurobipy as gp\nfrom gurobipy import GRB\navg_price={\'(865678,01:30,f)\': 898.85, \'(865678,01:30,s)\': 646.2, \'(865678,01:30,v)\': 494.22, \'(865678,01:30,l)\': 374.15, \'(915865,10:35,f)\': 995.75, \'(915865,10:35,s)\': 556.38, \'(915865,10:35,v)\': 411.27, \'(915865,10:35,l)\': 312.32}\nvalue_list={\'(865678,01:30,f)\': 0.028394, \'(865678,01:30,s)\': 0.05569, \'(865678,01:30,v)\': 0.045195, \'(865678,01:30,l)\': 0.042998, \'(915865,10:35,f)\': 0.018204, \'(915865,10:35,s)\': 0.021255, \'(915865,10:35,v)\': 0.017254, \'(915865,10:35,l)\': 0.030832}\nratio_list={\'(865678,01:30,f)\': 0.678261, \'(865678,01:30,s)\': 0.664991, \'(865678,01:30,v)\': 0.630688, \'(865678,01:30,l)\': 0.561085, \'(915865,10:35,f)\': 0.960781, \'(915865,10:35,s)\': 0.955126, \'(915865,10:35,v)\': 0.948269, \'(915865,10:35,l)\': 0.936171}\nvalue_0_list={\'865678\': 0.383933, \'915865\': 0.638646}\nratio_0_list={\'865678\': 1.783683, \'915865\': 1.025854}\navg_pax={\'865678\': 2279.18, \'915865\': 2294.45}\ncapacity_consum={\'f\': 2.0, \'s\': 1.6, \'v\': 1.3, \'l\': 1.0}\nflight_capacity={\'(865678,01:30)\': 589.96, \'(915865,10:35)\': 360.64}\n\nmodel = gp.Model("sales_based_lp")\nx = model.addVars(avg_price.keys(), lb=0, name="x") \nx_o = model.addVars(value_0_list.keys(), lb=0, name="x_o")  \n\nmodel.setObjective(gp.quicksum(avg_price[key] * x[key] for key in avg_price.keys()), GRB.MAXIMIZE)\n\n# dict() and string concatenation are used here instead of a brace literal and\n# an f-string, on purpose. This text passes through TWO brace-eating layers:\n# first the f-string that builds fewshot_example, then langchain\'s\n# PromptTemplate, which treats any surviving brace-name-brace as a prompt\n# variable and refuses to start the agent. Quadruple-escaping would work and\n# would be unreadable. The original template contains no literal braces at all,\n# and neither does this one -- including this comment, which sits inside the\n# f-string and would be evaluated like any other part of it.\n#\n# Keys look like "(<origin><destination>,<time>,<short>)", so everything before\n# the last comma identifies the flight and the last field is the fare family.\n# Group by flight, then sum that flight\'s families with their own consumption\n# coefficients against that flight\'s own capacity.\nflights = dict()\nfor key in avg_price.keys():\n    head, short = key.rsplit(\',\', 1)\n    flights.setdefault(head + \')\', []).append(key)\n\nfor flight_key, keys in flights.items():\n    model.addConstr(\n        gp.quicksum(capacity_consum[k.rsplit(\',\', 1)[1].rstrip(\')\')] * x[k] for k in keys)\n        <= flight_capacity[flight_key],\n        name="capacity_constraint_" + flight_key\n    )\n\nfor l in ratio_0_list.keys():\n    temp = 0\n    for key in ratio_list.keys():\n        if l in key:\n            temp += ratio_list[key] * x[key]\n    model.addConstr(\n        float(ratio_0_list[l]) * x_o[l] + temp == float(avg_pax[l])\n    )\n\nfor i in value_list.keys():\n    for l in ratio_0_list.keys():\n        if l in i:\n            model.addConstr(\n                value_0_list[l]  * x[i] <= value_list[i] * x_o[l]\n            )\n\nmodel.optimize()\n\nif model.status == GRB.OPTIMAL:\n    print("Optimal solution found:")\n    for v in model.getVars():\n        print(v.varName, v.x)\n    print("Optimal objective value:", model.objVal)\nelse:\n    print("No optimal solution found.")'

ESCAPED_FEWSHOT_CA = FEWSHOT_CA.replace('{', '{{').replace('}', '}}')

PREFIX_CA = (
    "Generate the SBLP mathematical model and Gurobi code from the current question and the attached raw CSV files. Read the files through the API; derive the parameters yourself. No custom data tools are available.\n\n"
    "Please refer to the following example and generate the answer in the same format:\n\n"
    + FEWSHOT_CA
    + "\n\n"
    + '    Note: Please retrieve all neccessary information from the CSV file to generate the answer.\n\n    The Generated Code section must be ONE python code block that runs on its own,\n    with nothing before it and nothing after it that it depends on. Inside that\n    block, every dictionary -- avg_price, value_list, ratio_list, value_0_list,\n    ratio_0_list, avg_pax, capacity_consum, flight_capacity -- must be written out\n    in full, with every key and every value as determined from the attached CSV files.\n\n    Do not abbreviate. Never replace a dictionary with an ellipsis placeholder, a\n    "see above" comment, a "same as before" comment, or a "truncated" comment. Do\n    not split the parameters into a separate code block from the model. Do not\n    repeat the parameters anywhere outside that one block. Length is not a reason\n    to shorten them: a block whose dictionaries are abbreviated cannot be executed\n    and counts as no answer at all.'
)

SUFFIX = """

Begin!

User Description: {input}
{agent_scratchpad}
"""


In [ ]:
def Fewshot_Only_CA(query: str) -> str:
    return API_FILES.answer(query, PREFIX_CA)


In [ ]:
# LARGE-SCALE NP/FLOW, few-shot-only ablation.
# The fixed NP exemplar remains; current-query numeric retrieval is replaced by
# the same API file attachments used by the CA half of this notebook.
from Test_Dataset.Air_NRM.large_scale.np_lean_workflow import install_np_workflow

_NP_WORKFLOW = install_np_workflow(globals(), ablation="fewshot_only")
globals().update(_NP_WORKFLOW)


## Combined function

In [ ]:
def Process_Input(query):
  """Route both large-scale CA and NP queries through few-shot-only agents.

  The classifier is not part of the ablation, so it is the classifier of
  LEAN_LLM_OPT_4.1_Air_NRM_LargeScale.ipynb, unchanged. Both branches keep a
  fixed worked example and receive API file attachments rather than retrieved values.
  """
  Type, output_model = None, None
  with lx.stage("classification"):
      category_original = Problemtype(query)
  print(f"Problem type classification finished, it belongs to {category_original}.")
  if "Sales-Based Linear Programming" in    category_original:
    print("Processing AirNRM queries")
    if "flow conservation constraints" in query or "flow conservation constraint" in query:
        print('----------Flow Constraints----------')
        print("Recommend Optimal Flights With Flow Conservation Constraints")
        with lx.stage("modeling"):
            output_model = ProcessPolicyFlow(query)
        Type = "Policy_Flow"

    else:
        print('----------CA----------')
        print("Only Develop Mathematic Formulations. No Recommendation for Flights.")
        with lx.stage("modeling"):
            output_model = Fewshot_Only_CA(query)
        Type = "CA"
  if output_model is None:
      raise ValueError(
          f"unroutable problem type: {category_original!r}")
  return Type, output_model


## Run the test

In [ ]:


import re as _re
from tqdm import tqdm


_FENCE_RE = _re.compile(r"```([A-Za-z0-9_+-]*)\n(.*?)```", _re.DOTALL)
# "avg_price = {...}  # (see above)" -- a set holding an Ellipsis, never real data.
_PLACEHOLDER_RE = _re.compile(r"^\s*\w+\s*=\s*\{\s*\.\.\.\s*\}")


def _find_dict_assignment(text, name):
    """Return the full `name = {...}` statement from anywhere in the answer.

    Brace-counted rather than regex-matched, because the dictionaries run to
    thousands of characters and may wrap. Placeholders are skipped, so this
    keeps looking until it finds a definition with real content.
    """
    for m in _re.finditer(rf"^{_re.escape(name)}\s*=\s*\{{", str(text), _re.M):
        depth = 0
        for i in range(m.end() - 1, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    statement = text[m.start(): i + 1]
                    if "..." not in statement:
                        return statement
                    break
    return None


def _extract_gurobipy_code(text):
    """Reassemble the gurobipy program from a model answer.

    The original returned the FIRST fenced block mentioning gurobi. That works
    while the answer is one block, which is what happens on the small instances.
    On the large ones gpt-4.1 splits the answer up: instance 14 came back as
    nine python fences -- one per parameter dict, then the model code -- and in
    that last fence the parameters were abbreviated to

        avg_price = {...}  # (see above)

    Returning only that fence loses every number and the code dies with
    "'set' object has no attribute 'keys'", because {...} is a set containing
    an Ellipsis. Scored as a modelling failure; it was not one. Reassembled by
    hand, that same answer solves to 7,558,440.98 against a label of
    7,558,440.98 -- 421 variables, 519 constraints, exact.

    So: concatenate the python fences in order, up to and including the last
    one that builds a model, and drop the placeholder redefinitions. Nothing is
    invented -- every number comes from a fence the model wrote.
    """
    if not text:
        return None
    blocks = [(lang.lower(), body) for lang, body in _FENCE_RE.findall(str(text))]
    python = [body for lang, body in blocks if lang in ("", "python", "py")]
    if not python:
        return None

    # Trailing fences are often sample output rather than program text.
    last = max((i for i, b in enumerate(python) if "gurobi" in b.lower()),
               default=len(python) - 1)

    lines = []
    placeholders = []
    for body in python[: last + 1]:
        for line in body.splitlines():
            hit = _PLACEHOLDER_RE.match(line)
            if hit:
                placeholders.append(line.split("=")[0].strip())
                continue
            lines.append(line)
    code = "\n".join(lines)

    # A placeholder whose real definition is nowhere in the fences is not lost:
    # the exemplar asks for the parameters twice -- once under "### Retrieved
    # Information" as plain markdown, once inside the code block -- and at 80
    # flights that is 26k tokens of the same dictionaries. The model reasonably
    # writes them out once, in the markdown, and abbreviates the second copy.
    # So look for the definition in the answer as a whole before giving up.
    recovered = []
    for name in placeholders:
        if _re.search(rf"^{_re.escape(name)}\s*=\s*\{{\s*['\"]", code, _re.M):
            continue                       # a real definition survived a fence
        statement = _find_dict_assignment(str(text), name)
        if statement:
            code = statement + "\n" + code
            recovered.append(name)

    code = code.strip()

    # Say so, loudly and greppably. The prompt now demands one self-contained
    # block; recovery firing at all means the model did not comply, and that is
    # a fact about the run, not a detail of the parser. Silently repairing it
    # would hide exactly the behaviour worth measuring.
    if placeholders:
        print(f"  [extract] FRAGMENTED-OUTPUT  blocks={last + 1} "
              f"placeholders={len(placeholders)} recovered={len(recovered)}"
              + ("" if len(recovered) == len(placeholders)
                 else "  <-- NOT fully recoverable"))

    # Every parameter the model template uses has to be bound to a real dict.
    # Without this the failure surfaces later as an opaque NameError or
    # "'set' object has no attribute 'keys'", which reads like a modelling
    # mistake rather than a truncated answer.
    required = ["avg_price", "value_list", "ratio_list", "value_0_list",
                "ratio_0_list", "avg_pax", "capacity_consum", "flight_capacity"]
    missing = [n for n in required
               if not _re.search(rf"^{n}\s*=\s*\{{\s*['\"]", code, _re.M)]
    if missing:
        print(f"  [extract] INCOMPLETE-CODE  no usable definition for "
              f"{', '.join(missing)} anywhere in the answer")

    return code or None


def Batch_Process_Queries(df, query_column='Query', log=LOG, cfg=CFG,
                          gold_column='Label-objective'):
    """
    Instrumented batch loop for the Air-NRM notebooks.

    Every query is wrapped in log.instance(...), so tokens / LLM calls /
    tool calls / cost / latency are recorded per instance and land in
    runs/<run_id>/instances.jsonl. Failures still produce a row, so the
    output frame always has the same length as the input frame.

    Two things beyond the original loop, both so the log is self-sufficient:
      * the ground-truth objective is carried through when the query file has
        one (see link_air_nrm_labels.py), otherwise score_runs.py has nothing
        to compare against;
      * the gurobipy block is extracted per instance rather than only in the
        run-everything cell downstream.
    """
    results = []
    has_gold = gold_column in df.columns

    for pos, (idx, row) in enumerate(tqdm(list(df.iterrows()),
                                          desc="Processing Queries")):
        query = row[query_column]
        # Label the instance by its position in the source file, not by
        # position in this slice -- ids have to stay comparable across batches.
        instance_id = int(idx) if isinstance(idx, (int,)) else pos
        gold = row[gold_column] if has_gold else None
        with log.instance(instance_id=instance_id, query=query,
                          gold_objective=gold) as rec:
            category, output_model = None, None
            try:
                # Record the instance so csv_qa_tool_CA can fall back to it for the
                # capacity terms when the agent's Action Input has dropped them.
                globals()['CURRENT_QUERY'] = query
                category, output_model = Process_Input(query)
                rec.set(pred_type=category, model_output=output_model,
                        code_output=_extract_gurobipy_code(output_model))
            except Exception as e:
                print(f"[{instance_id}] failed: {type(e).__name__}: {e}")
                rec.fail(e)                      # status -> "failed" in the log
        results.append({
            "Category": category,
            "Original_Query": query,
            "Output": output_model,
        })

    return pd.DataFrame(results)


In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import re
import numpy as np

def extract_objective(log_text):
   
    if "Optimal objective" in log_text:
        pattern = r"Optimal objective value: ([-+]?\d*\.?\d+)"
        match = re.search(pattern, log_text)
        if match:
            return float(match.group(1))

    elif "Best objective" in log_text:
        pattern = r"Best objective\s+([-+]?\d*\.?\d+e[-+]?\d+)"
        match = re.search(pattern, log_text, re.IGNORECASE)
        if match:
            return float(match.group(1))
    return None  



def gain_obj(df_out):
    output_code = df_out['Code'].tolist()
    obj = []
    i = 1

    for code in output_code:
        print(i)
        namespace = {'gp': gp, 'GRB': GRB}
        
  
        log_output = []
        

        class LogCapture:
            def write(self, message):
                log_output.append(message)

            def flush(self):  
                pass
            
        import sys
        log_capture = LogCapture()
        original_stdout = sys.stdout  
        sys.stdout = log_capture  

        try:
            exec(code, namespace)
        except Exception as e:
            log_output.append(f"Error: {e}\n")
            print(f"Error executing code block {i}: {e}")
        finally:
            sys.stdout = original_stdout 

    
        log_text = ''.join(log_output)

    
        optimal_value = extract_objective(log_text)
        if optimal_value is not None:
            obj.append(optimal_value)
            print(f"Optimal Value: {optimal_value}")
        else:

            obj.append("No optimal value found.")
            print("No optimal value found.")
        i += 1
    return obj


### Run CA

In [ ]:
# LARGE-SCALE, Abl-FewShotOnly: 15 instances, 112-421 variables each.
# run_notebook.py overrides this via --data / --rows.
df = pd.read_csv('Test_Dataset/Air_NRM/large_scale/query_CA.csv')
result_df = Batch_Process_Queries(df)
result_df.to_csv(LOG.dir / "FewshotOnly_answer_CA_largescale_GPT4.1.csv", index=False)

Code = []
for i in range(len(result_df['Output'])):
    text = result_df['Output'][i]
    code = _extract_gurobipy_code(text)
    Code.append(code if code else "No code found")

code_df = pd.DataFrame(Code, columns=['Code'])
code_df.to_csv(LOG.dir / "FewshotOnly_code_CA_largescale_GPT4.1.csv", index=False)

obj = gain_obj(code_df)
obj_df = pd.DataFrame({'Optimal Value': obj})
obj_df.to_csv(LOG.dir / "OBJ_CA_4.1_FewshotOnly.csv", index=False)

combined_df = pd.concat([result_df, code_df, obj_df], axis=1)
combined_df.to_csv(LOG.dir / "final_answer_CA_largescale_GPT4.1_FewshotOnly.csv", index=False)


### Run NP Flow

In [ ]:
# LARGE-SCALE NP/FLOW, Abl-FewShotOnly: 15 instances.
# run_notebook.py overrides this via --data / --rows.
df = pd.read_csv('Test_Dataset/Air_NRM/large_scale/query_NP_Flow.csv')
result_df = Batch_Process_Queries(df)
result_df.to_csv(LOG.dir / "FewshotOnly_answer_NP_Flow_largescale_GPT4.1.csv", index=False)

Code = []
for text in result_df['Output']:
    extracted = _extract_gurobipy_code(text)
    Code.append(extracted if extracted else "No code found")

code_df = pd.DataFrame(Code, columns=['Code'])
code_df.to_csv(LOG.dir / "FewshotOnly_code_NP_Flow_largescale_GPT4.1.csv", index=False)

obj = gain_obj(code_df)
obj_df = pd.DataFrame({'Optimal Value': obj})
obj_df.to_csv(LOG.dir / "OBJ_NP_4.1_FewshotOnly.csv", index=False)

combined_df = pd.concat([result_df, code_df, obj_df], axis=1)
combined_df.to_csv(LOG.dir / "final_answer_NP_Flow_largescale_GPT4.1_FewshotOnly.csv", index=False)


In [ ]:
API_FILES.close()
# ---------------------------------------------------------------
# Finish the run: writes runs/<run_id>/summary.json and prints the
# totals. Safe to run after every slice.
# ---------------------------------------------------------------
agg = LOG.close()

import json
rows = [json.loads(l) for l in open(LOG.dir_path / 'instances.jsonl')]
est = sum(r['summary'].get('n_estimated_token_calls', 0) for r in rows)
print(f"\ntoken source: {'provider (good)' if est == 0 else f'{est} calls ESTIMATED -- report as estimates'}")
print('run dir:', LOG.dir_path)
